In [ ]:
import csv
import os
import random
import re
import subprocess
import evaluate
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif as MIC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Lasso, ElasticNet
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import RFECV

# Seed for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Seed set to:", SEED)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Seed set to: 42
CUDA available: False
Device: cpu
Device name: CPU


# Load the data from Supabase

In [14]:
load_dotenv(Path.cwd().parent / ".env")

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_KEY"]
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

TABLE = "lablr"
PAGE_SIZE = 1000 

rows = []
start = 0
pages = 0
while True:
    resp = (
        supabase.table(TABLE)
        .select("*")
        .range(start, start + PAGE_SIZE - 1)
        .execute()
    )
    batch = resp.data
    if not batch:
        break
    rows.extend(batch)
    pages+=1
    if len(batch) < PAGE_SIZE:
        break
    start += PAGE_SIZE

df_lablr = pd.DataFrame(rows)
print(f"Pulled {len(df_lablr)} rows ({pages} pages) from '{TABLE}'")

Pulled 2187 rows (3 pages) from 'lablr'


In [5]:
df_lablr.head()

,id,entry_id,title,source_name,assigned_subsector,geography_scope,exec_summary,content,subsector_data,raw,approved,chosen_subsector,is_reclassified,reviewer_id,created_at
0,1365,094dcb0b643d0d82,Cybersecurity expert discusses situation at CHI,ketv.com,cyber_attack,NaN,CHI Health shut down some IT systems as a prec...,CHI Health shut down some IT systems as a prec...,"{'attack_type': 'cyber security issue', 'ranso...","{'id': '094dcb0b643d0d82', 'title': 'Cybersecu...",False,NaN,False,Briana,2026-06-30T22:28:14.056715+00:00
1,1564,0e1dd9a47b3f79f3,MI hospitals forced to send patients out of st...,wxyz.com,other,Michigan,Hospitals in Michigan are canceling surgeries ...,DETROIT (WXYZ) — Patients are being sent out o...,"{'severity': None, 'event_type': None, 'beds_o...","{'id': '0e1dd9a47b3f79f3', 'title': 'MI hospit...",False,NaN,False,Briana,2026-06-30T22:46:19.33946+00:00
2,1566,0ea76df689d90a4d,Hackers are extorting Globe Life with stolen c...,techcrunch.com,cyber_attack,US,Insurance giant Globe Life is being extorted b...,"Insurance giant Globe Life, which provides lif...","{'attack_type': 'extortion-only attack', 'rans...","{'id': '0ea76df689d90a4d', 'title': 'Hackers a...",True,cyber_attack,False,Briana,2026-06-30T22:46:39.642885+00:00
3,1605,01bedc1e3108ebbc,"US Fertility Sued Over Ransomware Attack, Heal...",healthitsecurity.com,cyber_attack,US,US Fertility (USF) has been sued by individual...,Getty Images\n\nUS Fertility (USF) has been su...,"{'attack_type': 'Ransomware', 'ransom_paid': F...","{'id': '01bedc1e3108ebbc', 'title': 'US Fertil...",True,cyber_attack,False,Evan,2026-07-01T15:57:22.943771+00:00
4,1610,0375ea5c1c6d5b7b,SE Health Center Of Stoddard County Announces ...,krcu.org,natural_disaster,Wisconsin,Southeast Health Center of Stoddard County is ...,"On July 28, the Southeast Health Center of Sto...","{'beds_offline': None, 'disaster_name': None, ...","{'id': '0375ea5c1c6d5b7b', 'title': 'SE Health...",False,NaN,False,Evan,2026-07-01T15:59:18.005658+00:00


### Consts and helper functions

In [ ]:
CLASS_NAMES = ["noise", "drug_shortage", "medical_device_shortage", "cyber_attack", "natural_disaster", "other"]

LABEL_MAP = {
    "noise": 0,
    "drug_shortage": 1,
    "medical_device_shortage": 2,
    "cyber_attack": 3,
    "natural_disaster": 4,
    "other": 5
}

IDX_TO_LABEL = {
    0: "noise",
    1: "drug_shortage",
    2: "medical_device_shortage",
    3: "cyber_attack",
    4: "natural_disaster",
    5: "other"
}

DROP_COLS = [
    "entry_id", "assigned_subsector", "raw", "chosen_subsector",
    "is_reclassified", "reviewer_id", "created_at", "approved"
]

X_COLS= ["title", "source_name", "content"]

def _make_pipe(clf) -> Pipeline:
    """
    Used to prep every model the same way before training

    Args:
        clf: the different classifiers used
    Returns:
        Pipeline object with the same specs
    """
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")), # swap Nan <-> median
        ("scale", StandardScaler()), # req for LR/SVC/MLP, RF doesn't need it but still gets it
        ("clf", clf),
    ])


def classic_models():
    """
    Returns:
        a fresh dict of NEW, untrained pipeline instances every call
    """
    return {
        "RandomForest": _make_pipe(RandomForestClassifier(random_state=SEED)),
        "LogisticRegression": _make_pipe(LogisticRegression(max_iter=1000, random_state=SEED)),
        "SVC": _make_pipe(SVC(random_state=SEED)),
        "MLP": _make_pipe(MLPClassifier(max_iter=1000, random_state=SEED)),
    }
# TODO: add more models here + bert 

def _derive_target(row):
    """
    Make a new label and class given a supabase row

    Args:
        row (straight from supabase)

    Returns:
        'label' and 'map' series with the correct classification
        If 'chosen_subsector' is not in LABEL_MAP returns nan pair
    """
    if not row["approved"]:
        return pd.Series({"label": 0, "class": "noise"})
    sub = row["chosen_subsector"]
    if sub in LABEL_MAP:
        return pd.Series({"label": LABEL_MAP[sub], "class": sub})
    return pd.Series({"label": np.nan, "class": np.nan}) 

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


df = df_lablr.copy()
df[["label", "class"]] = df.apply(_derive_target, axis=1) # axis 1 needed for cols

# drop unmapped edge-case rows, then lock label to int
df = df[df["label"].notna()].copy()
df["label"] = df["label"].astype(int) # make sure label is int
df = df.drop(columns=DROP_COLS)

X = df[X_COLS]
y = df["label"]

df["class"].value_counts()

class
cyber_attack               995
noise                      584
other                      281
drug_shortage              182
natural_disaster           115
medical_device_shortage     29
Name: count, dtype: int64

In [9]:
df.head()

,id,title,source_name,geography_scope,exec_summary,content,subsector_data,label,class
0,1365,Cybersecurity expert discusses situation at CHI,ketv.com,NaN,CHI Health shut down some IT systems as a prec...,CHI Health shut down some IT systems as a prec...,"{'attack_type': 'cyber security issue', 'ranso...",0,noise
1,1564,MI hospitals forced to send patients out of st...,wxyz.com,Michigan,Hospitals in Michigan are canceling surgeries ...,DETROIT (WXYZ) — Patients are being sent out o...,"{'severity': None, 'event_type': None, 'beds_o...",0,noise
2,1566,Hackers are extorting Globe Life with stolen c...,techcrunch.com,US,Insurance giant Globe Life is being extorted b...,"Insurance giant Globe Life, which provides lif...","{'attack_type': 'extortion-only attack', 'rans...",3,cyber_attack
3,1605,"US Fertility Sued Over Ransomware Attack, Heal...",healthitsecurity.com,US,US Fertility (USF) has been sued by individual...,Getty Images\n\nUS Fertility (USF) has been su...,"{'attack_type': 'Ransomware', 'ransom_paid': F...",3,cyber_attack
4,1610,SE Health Center Of Stoddard County Announces ...,krcu.org,Wisconsin,Southeast Health Center of Stoddard County is ...,"On July 28, the Southeast Health Center of Sto...","{'beds_offline': None, 'disaster_name': None, ...",0,noise


In [10]:
def run_experiment(X, y, models = classic_models()):
  """

  """
  results = {} # name = (mean_acc, std_acc)

  for name, pipe in models.items():
      print("="*75)
      print(name)

      # get a np array of scores
      scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
      results[name] = (scores.mean(), scores.std()) # save 1 score mean with std

      print(f"Accuracy per fold: {np.round(scores, 4)}")
      print(f"Mean accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

      #collects each window's out-of-fold prediction
      y_pred = cross_val_predict(pipe, X, y, cv=cv) # each window's out-of-fold prediction
      print("\nClassification report (out-of-fold):")
      print(classification_report(y, y_pred, target_names=CLASS_NAMES))

      # confusion matrix
      cm = confusion_matrix(y, y_pred, labels=range(len(CLASS_NAMES)))
      plt.figure(figsize=(5, 4))
      sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
      plt.title(f"{name} -- out-of-fold confusion matrix")
      plt.xlabel("Predicted")
      plt.ylabel("True")
      plt.show()
      print("="*75)


  print("Summary -- algorithms ranked by mean 5-fold accuracy")
  print("-"*60)
  ranking = sorted(results.items(), key=lambda kv: kv[1][0], reverse=True)
  for rank, (name, (mean_acc, std_acc)) in enumerate(ranking, start=1):
    print(f"{rank}. {name:<20} {mean_acc:.4f} ± {std_acc:.4f}")
    

In [ ]:
run_experiment(X,y)